# RAG-based Football Scouting System

This project implements a Retrieval-Augmented Generation (RAG) pipeline applied to football scouting using Transfermarkt data.

## Objective

The goal is to build a data-driven system capable of:
- identifying high-performing players
- generating player profiles from statistical data
- answering scouting-related questions using natural language

## Key Idea

Instead of using raw tabular data, we:
- aggregate player statistics (feature engineering)
- convert players into semantic textual profiles
- perform similarity search using embeddings (FAISS)
- generate answers using a language model (RAG)

## Motivation

This project highlights the complementarity between:
- structured data analysis (aggregation, ranking)
- unstructured reasoning (LLMs and natural language)

It demonstrates how machine learning systems can assist real-world decision-making in sports analytics.

In [73]:
# ===== Standard =====
import numpy as np
import pandas as pd

# ===== Embeddings =====
from sentence_transformers import SentenceTransformer

# ===== Vector search =====
!pip install faiss-cpu
import faiss

# ===== LLM / RAG =====
!pip install langchain-community
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_community.llms import Ollama

In [74]:
# ===== Embedding model =====
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ===== LLM =====
LLM_MODEL = "qwen2.5:0.5b"

# ===== Retrieval =====
TOP_K = 5

# ===== Data (optionnel pour debug) =====
MAX_PLAYERS = None  # mettre un nombre (ex: 10000) pour tester plus vite

## Data Loading

We use the Transfermarkt dataset, which contains detailed information about players, matches, and performances.

Due to the large size of the dataset (millions of rows), we load the data programmatically using KaggleHub to ensure reproducibility.

We keep the full dataset and perform filtering and aggregation later in the pipeline.

In [75]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("davidcariboo/player-scores")

print("Path to dataset files:", path)

# ===== Load full datasets =====

players = pd.read_csv(f"{path}/players.csv")
appearances = pd.read_csv(f"{path}/appearances.csv")
valuations = pd.read_csv(f"{path}/player_valuations.csv")

print("Players:", players.shape)
print("Appearances:", appearances.shape)
print("Valuations:", valuations.shape)

Using Colab cache for faster access to the 'player-scores' dataset.
Path to dataset files: /kaggle/input/player-scores
Players: (37579, 26)
Appearances: (1824008, 13)
Valuations: (526185, 6)


In [76]:
players.head()

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,IFM,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0


In [77]:
appearances["season"] = pd.to_datetime(appearances["date"]).dt.year

In [78]:
player_stats = (
    appearances
    .groupby(["player_id", "player_name", "season"])
    .agg({
        "goals": "sum",
        "assists": "sum",
        "yellow_cards": "sum",
        "red_cards": "sum"
    })
    .reset_index()
)

In [79]:
games = (
    appearances
    .groupby(["player_id", "season"])
    .size()
    .reset_index(name="games")
)

player_stats = player_stats.merge(games, on=["player_id", "season"], how="left")

In [80]:
# ===== Keep latest valuation per player =====

valuations_clean = (
    valuations
    .sort_values("player_id")
    .groupby("player_id")
    .tail(1)
)

player_stats = player_stats.merge(valuations_clean, on="player_id", how="left")

In [81]:
# ===== Scouting features =====

player_stats["goal_per_game"] = player_stats["goals"] / player_stats["games"]
player_stats["assist_per_game"] = player_stats["assists"] / player_stats["games"]

player_stats = player_stats.fillna(0)

In [82]:
player_stats.head()

,player_id,player_name,season,goals,assists,yellow_cards,red_cards,games,date,market_value_in_eur,current_club_name,current_club_id,player_club_domestic_competition_id,goal_per_game,assist_per_game
0,10,Miroslav Klose,2012,11,1,6,0,20,2007-06-21,23000000.0,SV Werder Bremen,398.0,IT1,0.550000,0.050000
1,10,Miroslav Klose,2013,9,4,3,0,29,2007-06-21,23000000.0,SV Werder Bremen,398.0,IT1,0.310345,0.137931
2,10,Miroslav Klose,2014,8,6,3,0,31,2007-06-21,23000000.0,SV Werder Bremen,398.0,IT1,0.258065,0.193548
3,10,Miroslav Klose,2015,12,8,6,0,36,2007-06-21,23000000.0,SV Werder Bremen,398.0,IT1,0.333333,0.222222
4,10,Miroslav Klose,2016,8,6,1,0,20,2007-06-21,23000000.0,SV Werder Bremen,398.0,IT1,0.400000,0.300000


In [83]:
# ===== Add competition_id to player_stats =====

comp = appearances[["player_id", "season", "competition_id"]].drop_duplicates()

player_stats = player_stats.merge(
    comp,
    on=["player_id", "season"],
    how="left"
)

In [84]:
# ===== League strength coefficients =====

TOP_LEAGUES = {
    "GB1": 1.0,
    "ES1": 0.95,
    "IT1": 0.9,
    "DE1": 0.9,
    "FR1": 0.85
}

player_stats["league_coeff"] = player_stats["competition_id"].map(TOP_LEAGUES).fillna(0.6)

In [85]:
# ===== Scoring function (performance + league level) =====

player_stats["scoring_score"] = (
    (player_stats["goal_per_game"] * 0.7 +
     player_stats["assist_per_game"] * 0.3)
    * np.log1p(player_stats["games"])
    * player_stats["league_coeff"]
)

In [86]:
player_stats[["competition_id", "league_coeff"]].head()

,competition_id,league_coeff
0,ELQ,0.6
1,IT1,0.9
2,EL,0.6
3,IT1,0.9
4,CIT,0.6


## Player Representation

To use machine learning models, we need to transform structured data into a format that can be processed semantically.

Each player-season is converted into a textual profile that summarizes:
- performance statistics (goals, assists, games)
- derived metrics (goals per game, assists per game)
- contextual information (market value)

This step is critical as it bridges structured tabular data and language-based models.

In [87]:
def player_to_text(row):
    return (
        f"Player {row['player_name']} in season {int(row['season'])}. "
        f"Goals: {int(row['goals'])}, Assists: {int(row['assists'])}, Games: {int(row['games'])}. "
        f"Goals per game: {row['goal_per_game']:.2f}. "
        f"Assists per game: {row['assist_per_game']:.2f}. "
        f"Market value: {int(row['market_value_in_eur'])} euros. "
        f"This player is considered a {'high goal scorer' if row['goal_per_game'] > 0.5 else 'moderate scorer' if row['goal_per_game'] > 0.2 else 'low scorer'}."
        f"League strength coefficient: {row['league_coeff']:.2f}. "
        f"This player plays in a {'top league' if row['league_coeff'] > 0.8 else 'lower tier league'}. "
    )

## Document Generation

We convert each player profile into natural language text.

This representation enables:
- semantic similarity search (via embeddings)
- compatibility with language models

The quality of this representation is crucial for the overall performance of the RAG system.

In [88]:
documents = player_stats.apply(player_to_text, axis=1).tolist()

In [89]:
documents[0]

'Player Miroslav Klose in season 2012. Goals: 11, Assists: 1, Games: 20. Goals per game: 0.55. Assists per game: 0.05. Market value: 23000000 euros. This player is considered a high goal scorer.League strength coefficient: 0.60. This player plays in a lower tier league. '

## Embedding Generation

We use a transformer-based model to convert textual player profiles into vector representations.

These embeddings capture semantic relationships between players, allowing us to:
- identify similar profiles
- retrieve relevant players given a query

In [ ]:
# ===== Embedding model =====

embedder = SentenceTransformer(EMBEDDING_MODEL)

# ===== Batch encoding (safe) =====

batch_size = 512
embeddings_list = []

for i in range(0, len(documents), batch_size):
    batch = documents[i:i+batch_size]
    emb = embedder.encode(batch)
    embeddings_list.append(emb)

embeddings = np.vstack(embeddings_list)

# ===== Normalize for cosine similarity =====
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
embeddings.shape

## Vector Database (FAISS)

We store embeddings in a FAISS index to enable efficient similarity search.

This allows fast retrieval of relevant player profiles based on a query.

In [ ]:
# ===== FAISS index =====

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)  # cosine similarity (grâce à normalisation)
index.add(embeddings)

In [ ]:
index.ntotal

## Retrieval

Given a query, we:
1. convert it into an embedding
2. search for similar player profiles
3. return the most relevant results

We also apply domain-specific filtering to improve relevance.

In [ ]:
class FAISSRetriever:
    def __init__(self, index, documents, k=TOP_K):
        self.index = index
        self.documents = documents
        self.k = k

    def retrieve(self, query):
        query_vec = embedder.encode([query])
        query_vec = query_vec / np.linalg.norm(query_vec, axis=1, keepdims=True)

        scores, indices = self.index.search(query_vec, self.k * 5)

        results = []
        for i in indices[0]:
            row = player_stats.iloc[i]

            # 🔥 filtre métier
            if row["goal_per_game"] > 0.3:
                threshold = player_stats["scoring_score"].quantile(0.85)

                if row["scoring_score"] > threshold:
                    results.append(self.documents[i])

            if len(results) >= self.k:
                break

        return results

In [ ]:
retriever = FAISSRetriever(index, documents)

results = retriever.retrieve("best goal scorers")

for r in results:
    print(r)
    print("-" * 50)

In [ ]:
prompt = PromptTemplate.from_template("""
You are a football scouting expert.

Answer the question using ONLY the context below.

Write a short and clear paragraph.

Do NOT write code.

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
def format_docs(docs):
    return "\n\n".join(docs)

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct",
    device=0
)

In [ ]:
def llm(prompt):
    prompt_str = str(prompt)

    output = generator(
        prompt_str,
        max_new_tokens=150,
        do_sample=False,
        return_full_text=False
    )

    text = output[0]["generated_text"]

    # 🔥 coupe dès que ça dérape
    stop_tokens = ["```", "def ", "The answer should"]
    for token in stop_tokens:
        if token in text:
            text = text.split(token)[0]

    return text.strip()

## RAG Pipeline

We combine:
- retrieved player profiles (context)
- a language model

The model generates answers based only on the retrieved context.

This enables natural language interaction with structured football data.

In [ ]:
retriever = FAISSRetriever(index, documents)

rag_chain = (
    {
        "context": lambda x: format_docs(retriever.retrieve(x)),
        "question": lambda x: x
    }
    | prompt
    | RunnableLambda(llm)
)

In [ ]:
rag_chain.invoke("Describe the profile of a better player than Kolo Muani")

## Analysis & Future Improvements

### Analysis of Results

The RAG system is able to generate coherent and interpretable scouting insights based on player statistics.

It successfully:
- identifies relevant offensive players based on performance metrics
- highlights key indicators such as goals and assists per game
- produces natural language explanations from structured data

However, the quality of the results strongly depends on:
- the design of the scoring function
- the relevance of retrieved players
- the richness of the features used to represent player performance

This confirms that retrieval quality is the most critical component in a RAG system applied to structured data.

---

### Limitations

Several limitations remain:

- The system relies on aggregated averages, which can hide variability in performance  
- It does not account for contextual factors such as:
  - strength of opponents
  - match importance
- It does not model player consistency over time  
- Some leagues may still be under- or over-valued despite the applied coefficients  

Additionally, computational cost can become significant when:
- scaling to larger datasets  
- using more advanced embedding or language models  

---

### Possible Improvements

This system could be significantly improved by incorporating richer football-specific features:

- **Age & potential**  
  → distinguish between peak players and high-potential young talents  

- **Player trajectory (career history)**  
  → detect progression or decline over seasons  

- **Position-specific analysis**  
  → adapt evaluation metrics depending on the role (forward, midfielder, defender)  

- **Consistency metrics**  
  → go beyond averages and measure regularity of performance  

- **Performance against top teams**  
  → better evaluate players under high competitive pressure  

- **Advanced scouting metrics**  
  → expected goals (xG), expected assists (xA), shot quality, etc.  

---

### Key Takeaway

This project highlights an important insight:

> RAG systems are powerful for interpreting data, but require strong domain-specific preprocessing to ensure meaningful results.

Rather than replacing structured analysis, RAG complements it by enabling intuitive and explainable interaction with data.

This hybrid approach is particularly relevant for real-world decision support systems.